In [ ]:
# Notebook path bootstrap
from pathlib import Path
import sys

REPO_ROOT = Path("./LINCS_scGPT_embeddings")
METADATA_DIR = REPO_ROOT / "Data" / "metadata"
LINCS_DIR = REPO_ROOT / "Data" 

# Build adata object from LINCS Level 3 data

Data can be downloaded from https://clue.io/data/CMap2020#LINCS2020

In [ ]:
# IMPORT NEEDED PACKAGES 
import pandas as pd
import numpy as np

from tqdm import tqdm
import pickle
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
from cmapPy.pandasGEXpress.parse import parse
import scanpy as sc


In [ ]:
# Load metadata 
root_LINCS_meta = f"{METADATA_DIR}/" 
root_LINCS = f"{LINCS_DIR}/"
cp_info = pd.read_csv(root_LINCS_meta + "compoundinfo_beta.txt", sep="\t", dtype=str)
cell_info = pd.read_csv(root_LINCS_meta + "cellinfo_beta.txt", sep="\t", dtype=str)
inst_info = pd.read_csv(root_LINCS_meta + "instinfo_beta.txt", sep="\t", dtype=str)
sig_info = pd.read_csv(root_LINCS_meta + "siginfo_beta.txt", sep = "\t", dtype=str)
gene_info = pd.read_csv(root_LINCS_meta + "geneinfo_beta.txt", sep="\t", dtype=str)

In [ ]:
sh_data = parse(root_LINCS + 'level3_beta_trt_sh_n453175x12328.gctx', cid = inst_info[inst_info.pert_type == 'trt_sh'].sample_id.unique(), rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
cmp_data = parse(root_LINCS + 'level3_beta_trt_cp_n1805898x12328.gctx', rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
oe_data = parse(root_LINCS + 'level3_beta_trt_oe_n131668x12328.gctx',rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
xpr_data = parse(root_LINCS + 'level3_beta_trt_xpr_n420583x12328.gctx', rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
ctr_data = parse(root_LINCS + 'level3_beta_ctl_n188708x12328.gctx', rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
misc_data = parse(root_LINCS + 'level3_beta_trt_misc_n26428x12328.gctx', rid=gene_info[gene_info.feature_space == 'landmark'].gene_id.unique())

In [ ]:
sh_data = sh_data.data_df.sort_index()
cmp_data = cmp_data.data_df.sort_index()
oe_data = oe_data.data_df.sort_index()
xpr_data = xpr_data.data_df.sort_index()
ctr_data = ctr_data.data_df.sort_index()
misc_data = misc_data.data_df.sort_index()

In [ ]:
data = pd.concat([sh_data, cmp_data, oe_data, xpr_data, ctr_data, misc_data], axis = 1)
del(sh_data, cmp_data, oe_data, xpr_data, ctr_data, misc_data)

In [ ]:
gene_info.gene_id = gene_info.gene_id.astype(str)

In [ ]:
adata = sc.AnnData(X = data.T.sort_index(), obs = inst_info.set_index('sample_id').sort_index(), var = gene_info[gene_info.gene_id.isin(data.index)].set_index('gene_id').sort_index())

In [ ]:
inst_info.sort_values(by='sample_id', inplace=True)

In [ ]:
adata.obs = adata.obs.merge(inst_info, left_index=True, right_on='sample_id', how='left')
adata.obs.index = adata.obs['sample_id']
adata.obs.sample_id = adata.obs.sample_id.astype(str)
adata.obs.build_name = adata.obs.build_name.astype(str)

In [ ]:
min_val = adata.X.min()
if min_val < 0:
    print(f"Clipping negative values (min={min_val}) to zero.")
    adata.X = np.maximum(adata.X, 0)

adata_lincs = adata[adata.X.sum(axis=1) > 0].copy()    

In [ ]:
adata.var['gene_name'] = adata.var.gene_symbol

In [ ]:
adata_lincs.write_h5ad(root_LINCS + 'LINCS_2020_adata.h5ad')

## HQ adata

In [ ]:
# use signature info to filter out low quality data 
filtered_data = sig_info[
    (sig_info['qc_pass'] == '1') &
    (sig_info['is_hiq'] == '1') &
    (sig_info['cc_q75'].astype(float) > 0.2) &
    (sig_info['tas'].astype(float) >= 0.2) &
    (sig_info['batch_effect_tstat_pct'].astype(float) < 95)]

In [ ]:
# obtain the inst ids for the filtered data
distil_ids = filtered_data['distil_ids']
instance_ids = distil_ids.str.split('|')
flat_instance_ids = [inst for sublist in instance_ids for inst in sublist]
ins_sel = inst_info[inst_info.sample_id.isin(flat_instance_ids)]
ins_sel = ins_sel[ins_sel['qc_pass'] == '1']

In [ ]:
sig_info_conversion = filtered_data[['sig_id', 'distil_ids']]
sig_info_conversion['distil_ids'] = sig_info_conversion['distil_ids'].str.split('|')
sig_info_conversion = sig_info_conversion.explode('distil_ids')

In [ ]:
distil2sig = dict(zip(sig_info_conversion['distil_ids'], sig_info_conversion['sig_id']))
sig2tas = dict(zip(filtered_data['sig_id'], filtered_data['tas']))

In [ ]:
list_index_adata = adata.obs.index
tas = []
for i in tqdm(list_index_adata): 
    if i in distil2sig.keys(): 
        tas.append(sig2tas[distil2sig[i]])
    else:
        tas.append('nan')

In [ ]:
adata.obs['tas'] = tas
adata.obs['tas'] = adata.obs['tas'].astype(str)

In [ ]:
# Convert all columns in obs to strings if needed
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

# Convert all columns in var to strings if needed
for col in adata.var.columns:
    if adata.var[col].dtype == 'object':
        adata.var[col] = adata.var[col].astype(str)

In [ ]:
adata.write_h5ad(root_LINCS + 'LINCS_2020_filtered.h5ad')